# Installation

In [1]:
!pip -q uninstall -y transformers
!pip -q install "transformers==4.41.0" datasets evaluate tensorflow tf-keras

# Imports + version check

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
import transformers
from datasets import load_dataset

print("Tensorflow:", tf.__version__)
print("Transformers:", transformers.__version__)

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification


/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tensorflow: 2.12.0
Transformers: 4.41.0


# Load dataset (IMDB)

In [3]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

# Use a subset (faster) or set to full

In [4]:
train_size = 20000
test_size = 5000

train_ds = dataset["train"].shuffle(seed=42).select(range(train_size))
test_ds = dataset["test"].shuffle(seed=42).select(range(test_size))

print("Train:", len(train_ds), "Test", len(test_ds))
print(train_ds[0])

Train: 20000 Test 5000
{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


# Tokenize

In [5]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])


# Convert to tf.data.Dataset

In [6]:
batch_size = 16

tf_train = train_tok.to_tf_dataset(
    columns=["input_ids", "token_type_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=True,
    batch_size=batch_size
)

tf_test = test_tok.to_tf_dataset(
    columns=["input_ids", "token_type_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=False,
    batch_size=batch_size
)

/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/datasets/arrow_dataset.py:403: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


# Load TF BERT model

In [7]:
model = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.summary()

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109,483,778
Trainable params: 109,483,778
Non-trainable params: 0
_________________________________________________________________


# Compile + Train (Keras)

In [8]:
epochs = 2
lr = 2e-5

optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy("accuracy")]

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

history = model.fit(tf_train, validation_data=tf_test, epochs=epochs)

Epoch 1/2


AttributeError: in user code:

    File "/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/keras/engine/training.py", line 1284, in train_function  *
        return step_function(self, iterator)
    File "/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/keras/engine/training.py", line 1268, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/keras/engine/training.py", line 1249, in run_step  **
        outputs = model.train_step(data)
    File "/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/transformers/modeling_tf_utils.py", line 1630, in train_step
        x, y, sample_weight = keras.utils.unpack_x_y_sample_weight(data)

    AttributeError: module 'keras.utils' has no attribute 'unpack_x_y_sample_weight'


# Evaluate

In [ ]:
results = model.evaluate(tf_test, verbose=1)
print("Test Loss, Test Accuracy:", results)

# Predict custom sentences

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
def predict_sentiment(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="tf")
    logits = model(enc).logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    for t, p, pr in zip(texts, preds, probs):
        print(f"\nText: {t[:120]}{'...' if len(t)>120 else ''}")
        print("Prediction:", id2label[int(p)], "| Prob:", pr)

predict_sentiment([
    "This movie was absolutely fantastic, I loved it."
    "Worst film ever. Boring and pointless."
])